In [5]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
import itertools

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_scatter import scatter_max, scatter_mean
from torch_scatter import scatter
from typing import Tuple, Optional, Dict, List, Sequence

In [11]:
from torch_pointcloud.layers.blocks import linear_block

linear_block(3, 64, act=nn.ReLU(), norm=nn.BatchNorm1d(64), bias=True)

Sequential(
  (0): Linear(in_features=3, out_features=64, bias=True)
  (1): ReLU()
  (2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (3): Dropout(p=0.0, inplace=False)
)

In [ ]:
class TNet(nn.Module):
    def __init__(
        self,
        k: int = 3,
        mlp1_dims: Sequence[int] = (64, 128, 1024),
        mlp2_dims: Sequence[int] = (512, 256),
        global_pool: str = "max",
    ) -> None:
        super().__init__()
        self.k = k
        self.mlp1_dims = mlp1_dims
        self.mlp2_dims = mlp2_dims
        self.global_pool = global_pool

        blocks = []
        for in_features, out_features in itertools.pairwise([k] + mlp1_dims):
            blocks.append(linear_block(in_features, out_features, dropout=None, order="lan"))
        self.mlp1 = nn.Sequential(*blocks)

        blocks = []
        for in_features, out_features in itertools.pairwise([mlp1_dims[-1]] + mlp2_dims):
            blocks.append(linear_block(in_features, out_features, dropout=None, order="lan"))
        self.mlp2 = nn.Sequential(*blocks)

        self.transform = nn.Linear(mlp2_dims[-1], k * k)
        nn.init.zeros_(self.transform.weight)
        nn.init.eye_(self.transform.bias.view(k, k))

    def forward(self, x: torch.Tensor, batch_idxs: torch.Tensor) -> torch.Tensor:
        x = self.mlp1(x)
        x = scatter(x, batch_idxs, dim=0, reduce=self.global_pool)
        x = self.mlp2(x)

        x = self.transform(x)
        iden = torch.eye(self.k, dtype=x.dtype, device=x.device)
        x = x.view(-1, self.k, self.k) + iden

        return x[batch_idxs]

In [72]:
import itertools


for in_features, out_features in itertools.pairwise([3, 64, 128, 1024]):
    print(in_features, out_features)

3 64
64 128
128 1024


In [70]:
coords = torch.randn(1000, 3)     # XYZ coordinates
rgb = torch.rand(1000, 3)         # RGB values
# Batch of two point clouds
batch = torch.tensor([0] * 600 + [1] * 400, dtype=torch.long)

tnet = TNet()
tnet(coords, batch)

tensor([[[2., 0., 0.],
         [0., 2., 0.],
         [0., 0., 2.]],

        [[2., 0., 0.],
         [0., 2., 0.],
         [0., 0., 2.]],

        [[2., 0., 0.],
         [0., 2., 0.],
         [0., 0., 2.]],

        ...,

        [[2., 0., 0.],
         [0., 2., 0.],
         [0., 0., 2.]],

        [[2., 0., 0.],
         [0., 2., 0.],
         [0., 0., 2.]],

        [[2., 0., 0.],
         [0., 2., 0.],
         [0., 0., 2.]]], grad_fn=<IndexBackward0>)

In [71]:
tnet

TNet(
  (mlp1): Sequential(
    (0): Sequential(
      (0): Linear(in_features=3, out_features=64, bias=True)
      (1): ReLU()
      (2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): Sequential(
      (0): Linear(in_features=64, out_features=128, bias=True)
      (1): ReLU()
      (2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (2): Sequential(
      (0): Linear(in_features=128, out_features=1024, bias=True)
      (1): ReLU()
      (2): BatchNorm1d(1024, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
  )
  (mlp2): Sequential(
    (0): Sequential(
      (0): Linear(in_features=1024, out_features=512, bias=True)
      (1): ReLU()
      (2): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): Sequential(
      (0): Linear(in_features=512, out_features=256, bias=True)
      (1): ReLU()
      (2): BatchNorm1d(256, eps=1e-05, mo